# 03 · Gold — Power BI için yıldız şema (Direct Lake)

| Tablo | Tanecik (grain) | Kullanım |
|---|---|---|
| `gold_fact_cost_daily` | gün × abonelik × kaynak × servis × meter × charge/pricing kategorisi | Tüm maliyet ölçüleri |
| `gold_dim_date` | gün (tam takvim yılları) | Zaman zekâsı: MTD, YTD, MoM |
| `gold_dim_subscription` | abonelik | Abonelik adı, fatura hesabı |
| `gold_dim_resource` | kaynak (`resource_key`) | Kaynak grubu, bölge, etiketler |
| `gold_dim_service` | servis | Servis kategorisi |
| `gold_cost_monthly` | ay × abonelik × servis | Hazır MoM / YTD tablosu |

Kaynağı olmayan satırlar (rezervasyon satın alma, destek planı…) `resource_key = 'unassigned/<abonelik-id>'` alır,
böylece ilişkiler kırılmaz, her abonelikte ayrı görünür ve toplamlar Silver ile birebir tutar.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

MONEY = DecimalType(18, 4)
UNASSIGNED = "unassigned"

silver = (
    spark.read.format("delta").load("Tables/dbo/silver_costs")
    .withColumn("subscription_id", F.coalesce("subscription_id", F.lit("unknown")))
    .withColumn(
        "resource_key",
        F.coalesce("resource_id", F.concat(F.lit(UNASSIGNED + "/"), F.col("subscription_id"))),
    )
)


def write(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"Tables/dbo/{name}")
    print(f"{name:<22} {spark.read.format('delta').load(f'Tables/dbo/{name}').count():>8,} satır")

## Fact: günlük maliyet

In [ ]:
fact = (
    silver.groupBy(
        "charge_date", "subscription_id", "resource_key", "service_name", "meter_category",
        "charge_category", "pricing_category", "billing_currency",
    )
    .agg(
        F.sum("billed_cost").cast(MONEY).alias("billed_cost"),
        F.sum("effective_cost").cast(MONEY).alias("effective_cost"),
        F.sum("list_cost").cast(MONEY).alias("list_cost"),
        F.sum("consumed_quantity").alias("consumed_quantity"),
        F.count("*").alias("line_items"),
    )
    .withColumn("savings", (F.col("list_cost") - F.col("effective_cost")).cast(MONEY))
)
write(fact, "gold_fact_cost_daily")

## Boyutlar: abonelik, kaynak, servis, tarih

In [ ]:
latest_sub = Window.partitionBy("subscription_id").orderBy(F.desc("charge_date"))
dim_subscription = (
    silver.withColumn("_rn", F.row_number().over(latest_sub)).filter("_rn = 1")
    .select(
        "subscription_id",
        F.coalesce("subscription_name", "subscription_id").alias("subscription_name"),
        F.coalesce("billing_account_name", F.lit("(not set)")).alias("billing_account_name"),
    )
)
write(dim_subscription, "gold_dim_subscription")

latest_first = Window.partitionBy("resource_key").orderBy(F.desc("charge_date"), F.desc("effective_cost"))
is_unassigned = F.col("resource_key").startswith(UNASSIGNED)

dim_resource = (
    silver.withColumn("_rn", F.row_number().over(latest_first)).filter("_rn = 1")
    .select(
        "resource_key", "resource_id",
        F.when(is_unassigned, "(no resource)").otherwise(F.coalesce("resource_name", F.lit("(not set)"))).alias("resource_name"),
        "resource_type",
        F.when(is_unassigned, "(no resource group)").otherwise(F.coalesce("resource_group", F.lit("(not set)"))).alias("resource_group"),
        "subscription_id", "subscription_name", "region",
        F.coalesce("tag_environment", F.lit("(not set)")).alias("tag_environment"),
        F.coalesce("tag_cost_center", F.lit("(not set)")).alias("tag_cost_center"),
        F.coalesce("tag_project", F.lit("(not set)")).alias("tag_project"),
        "tags",
    )
)
write(dim_resource, "gold_dim_resource")

dim_service = silver.groupBy("service_name").agg(F.first("service_category", ignorenulls=True).alias("service_category"))
write(dim_service, "gold_dim_service")

# Full calendar years keep DAX time intelligence (TOTALYTD, DATEADD) correct.
bounds = silver.agg(
    F.make_date(F.year(F.min("charge_date")), F.lit(1), F.lit(1)).alias("start"),
    F.make_date(F.year(F.max("charge_date")), F.lit(12), F.lit(31)).alias("end"),
)
dim_date = (
    bounds.select(F.explode(F.sequence("start", "end")).alias("date"))
    .select(
        "date",
        F.year("date").alias("year"),
        F.concat(F.lit("Q"), F.quarter("date")).alias("quarter"),
        F.month("date").alias("month"),
        F.date_format("date", "MMM").alias("month_name"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        (F.year("date") * 100 + F.month("date")).alias("year_month_sort"),
        F.dayofmonth("date").alias("day"),
        ((F.dayofweek("date") + 5) % 7 + 1).alias("day_of_week"),   # Monday = 1
        F.date_format("date", "E").alias("day_name"),
        F.dayofweek("date").isin(1, 7).alias("is_weekend"),
    )
)
write(dim_date, "gold_dim_date")

## Aylık özet: MoM ve YTD

In [ ]:
monthly = (
    silver.withColumn("year_month", F.date_format("charge_date", "yyyy-MM"))
    .groupBy("year_month", "subscription_id", "service_name")
    .agg(F.sum("effective_cost").alias("effective_cost"), F.sum("list_cost").alias("list_cost"))
)

# Previous month is matched by calendar month (not row order), so gaps yield NULL.
previous = monthly.select(
    F.date_format(F.add_months(F.to_date(F.concat("year_month", F.lit("-01"))), 1), "yyyy-MM").alias("year_month"),
    "subscription_id",
    "service_name",
    F.col("effective_cost").alias("previous_month_cost"),
)
ytd_window = (
    Window.partitionBy("subscription_id", "service_name", F.substring("year_month", 1, 4))
    .orderBy("year_month")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

gold_monthly = (
    monthly.join(previous, ["year_month", "subscription_id", "service_name"], "left")
    .withColumn("mom_change", F.col("effective_cost") - F.col("previous_month_cost"))
    .withColumn("mom_change_pct", F.round(F.col("mom_change") / F.col("previous_month_cost"), 4))
    .withColumn("ytd_cost", F.sum("effective_cost").over(ytd_window))
    .select(
        "year_month", "subscription_id", "service_name",
        *[F.col(c).cast(MONEY).alias(c) for c in ("effective_cost", "list_cost", "previous_month_cost", "mom_change")],
        "mom_change_pct",
        F.col("ytd_cost").cast(MONEY).alias("ytd_cost"),
    )
)
write(gold_monthly, "gold_cost_monthly")

## Mutabakat: Gold toplamı = Silver toplamı

In [ ]:
silver_total = silver.agg(F.sum("effective_cost")).first()[0] or 0
gold_total = spark.read.format("delta").load("Tables/dbo/gold_fact_cost_daily").agg(F.sum("effective_cost")).first()[0] or 0
diff = abs(float(gold_total) - float(silver_total))
print(f"Silver: {silver_total:,.4f}   Gold: {float(gold_total):,.4f}   fark: {diff:.4f}")
assert diff < 0.05, "Gold ve Silver toplamları uyuşmuyor!"